# 19 · Shifts & Monitoring — The World Is Not i.i.d.

**Part 2 · Machine Learning Systems** · *Prerequisites: 17–18* · *Book mapping: DMLS ch. 8*

Classical software fails loudly — exceptions, 500s, crashes. A model fails **silently**: it
keeps returning confident numbers while the world it learned drifts away underneath. The
model didn't change; the *data* did. This notebook builds a drifting world, deploys a model
into it, and constructs the monitoring that notices — including the case where nothing can
notice except the labels, and the case where the model is *causing* the drift itself.

The taxonomy (train distribution P(X, Y) vs serving distribution):

- **Covariate shift**: P(X) changes, P(Y|X) intact — different *inputs*, same rules.
  (New marketing channel brings different users.)
- **Label shift**: P(Y) changes — prevalence moves. (Fraud season.)
- **Concept drift**: **P(Y|X) changes** — the rules themselves. (A pandemic redefines
  "normal purchasing".) Sudden, gradual, or recurring (seasonality).
- And the impostor that outnumbers them all in practice: **pipeline failure** — an
  upstream job starts emitting nulls (nb 14's audit exists for this). Check the plumbing
  before theorizing about the world.

**Learning objectives.**

1. Simulate covariate shift and concept drift; watch what each does to accuracy.
2. Build three input-drift detectors — **KS test**, **PSI**, **domain classifier** — and
   find each one's blind spot (including the multivariate shift only the classifier sees).
3. Prove the hard truth: pure concept drift is invisible to *every* label-free monitor —
   and design around label delay.
4. Build a monitoring panel with alarms that fire at the right days and stay quiet
   otherwise.
5. Simulate a **degenerate feedback loop** (the popularity spiral) and break it with
   exploration.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import ks_2samp
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

from sysdes import viz

viz.use_style()
rng = np.random.default_rng(seed=19)

## 1. A world that drifts, on schedule

120 days of traffic, 400 requests/day. The true label depends on x₀, x₁ with a mild
curvature. Two planned events: **day 40** — covariate shift (a new acquisition channel
moves the input distribution); **day 80** — concept drift (the relationship itself
rotates). A logistic model is trained on days 0–29 and never retrained:

In [ ]:
DAYS, PER_DAY = 120, 400

def sample_day(day: int, r) -> tuple[np.ndarray, np.ndarray]:
    # Covariate shift at day 40: the input cloud moves off-center.
    center = np.array([0.0, 0.0]) if day < 40 else np.array([1.0, -0.5])
    X = r.normal(0, 1, (PER_DAY, 2)) + center
    # Concept drift at day 80: the boundary rotates.
    if day < 80:
        logits = X[:, 0] + 0.6 * X[:, 1] - 0.15 * X[:, 0] ** 2 - 0.3
    else:
        logits = -0.4 * X[:, 0] + 1.1 * X[:, 1] - 0.15 * X[:, 0] ** 2 - 0.3
    y = (logits + r.normal(0, 0.6, PER_DAY) > 0).astype(int)
    return X, y

world = [sample_day(d, np.random.default_rng(1000 + d)) for d in range(DAYS)]
X_train = np.vstack([X for X, _ in world[:30]])
y_train = np.concatenate([y for _, y in world[:30]])
model = LogisticRegression().fit(X_train, y_train)

true_acc = np.array([model.score(X, y) for X, y in world])
print(f"accuracy - days 0-39: {true_acc[:40].mean():.3f}, "
      f"days 40-79 (covariate shift): {true_acc[40:80].mean():.3f}, "
      f"days 80+ (concept drift): {true_acc[80:].mean():.3f}")
assert true_acc[40:80].mean() < true_acc[:40].mean() - 0.02, \
    "covariate shift hurt some: the model is least trained where the inputs moved"
assert true_acc[80:].mean() < 0.62, "concept drift is the catastrophe"

Note the asymmetry the numbers show: covariate shift *may* degrade you (it moved traffic
into a region the model approximates worse) but the rules held; concept drift broke the
rules. Now — could monitoring have seen these coming, and *when*?

## 2. Input-drift detectors (no labels required)

Labels arrive late (§3); inputs arrive instantly. Input monitors compare a **reference
window** (training data) against each day's traffic.

In [ ]:
reference = X_train

def psi(ref: np.ndarray, cur: np.ndarray, bins: int = 10) -> float:
    """Population Stability Index for one feature: sum over bins of
    (cur% - ref%) * ln(cur% / ref%). Rules of thumb: <0.1 stable,
    0.1-0.25 investigate, >0.25 shifted."""
    edges = np.quantile(ref, np.linspace(0, 1, bins + 1))
    edges[0], edges[-1] = -np.inf, np.inf
    ref_pct = np.histogram(ref, edges)[0] / len(ref) + 1e-6
    cur_pct = np.histogram(cur, edges)[0] / len(cur) + 1e-6
    return float(np.sum((cur_pct - ref_pct) * np.log(cur_pct / ref_pct)))

ks_pvals = np.array([[ks_2samp(reference[:, j], X[:, j]).pvalue for j in (0, 1)]
                     for X, _ in world])
psi_scores = np.array([[psi(reference[:, j], X[:, j]) for j in (0, 1)]
                       for X, _ in world])

# Alarm rule: any feature's KS p-value below 0.01/2 (Bonferroni for 2 features).
input_alarms = np.where((ks_pvals < 0.01 / 2).any(axis=1))[0]
first_input_alarm = int(input_alarms[input_alarms >= 30].min())
print(f"first input-drift alarm: day {first_input_alarm} "
      f"(PSI that day: {psi_scores[first_input_alarm].round(2)})")
assert first_input_alarm == 40, "the covariate shift is caught THE DAY it happens"
assert (ks_pvals[30:40] > 0.01 / 2).all(), "and the stable weeks raise no false alarms"
assert psi_scores[45:80, 0].mean() > 0.25, "PSI agrees: 'shifted' by industry rule"

Instant detection, zero labels. But per-feature tests have a structural blind spot:
they check **marginals**. A shift in the *relationship between* features — correlation
flips, new co-occurrence patterns — leaves every marginal untouched. The detector for
that is beautifully self-referential: **train a classifier to distinguish reference from
current** (a *domain classifier*). If it can (AUC ≫ 0.5), the distributions differ —
in whatever dimension matters:

In [ ]:
cov_ref = np.array([[1, 0.8], [0.8, 1]])
cov_cur = np.array([[1, -0.8], [-0.8, 1]])            # correlation FLIPS...
ref_c = rng.multivariate_normal([0, 0], cov_ref, 3_000)
cur_c = rng.multivariate_normal([0, 0], cov_cur, 3_000)
# ...while each feature's marginal stays exactly N(0,1):

p0 = ks_2samp(ref_c[:, 0], cur_c[:, 0]).pvalue
p1 = ks_2samp(ref_c[:, 1], cur_c[:, 1]).pvalue
psi_max = max(psi(ref_c[:, j], cur_c[:, j]) for j in (0, 1))

X_dom = np.vstack([ref_c, cur_c])
y_dom = np.r_[np.zeros(len(ref_c)), np.ones(len(cur_c))]
shuffle = rng.permutation(len(X_dom))
dom = HistGradientBoostingClassifier(random_state=0).fit(
    X_dom[shuffle][:4000], y_dom[shuffle][:4000])
dom_auc = roc_auc_score(y_dom[shuffle][4000:],
                        dom.predict_proba(X_dom[shuffle][4000:])[:, 1])

print(f"correlation flip: KS p-values ({p0:.2f}, {p1:.2f}) - blind; "
      f"max PSI {psi_max:.3f} - blind; domain-classifier AUC {dom_auc:.2f} - CAUGHT")
assert p0 > 0.05 and p1 > 0.05 and psi_max < 0.1, "marginal tests see nothing"
assert dom_auc > 0.75, "the domain classifier sees the joint distribution"

(Bonus: the domain classifier's feature importances tell you *what* shifted — a debugging
gift no p-value provides.)

## 3. The hard truth about concept drift

Day 80 changed P(Y|X) with P(X) untouched. Walk the monitor lineup: input tests — see
X only, silent. Prediction-distribution monitors — the model is a fixed function of X,
so if X's distribution didn't move, *predictions didn't either*. Silent. Verify both:

In [ ]:
pred_mean = np.array([model.predict_proba(X)[:, 1].mean() for X, _ in world])

# Around day 80 (the catastrophe): inputs and predictions both look serene.
# (After investigating the day-40 alarm, ops re-baselined the input monitor to
# the new normal - so compare days 80-99 against days 60-79:)
rebase_ref = np.vstack([X for X, _ in world[60:80]])
rebase_cur = np.vstack([X for X, _ in world[80:100]])
ks_at_80 = min(ks_2samp(rebase_ref[:, j], rebase_cur[:, j]).pvalue for j in (0, 1))
pred_jump_40 = abs(pred_mean[40:50].mean() - pred_mean[30:40].mean())
pred_jump_80 = abs(pred_mean[80:90].mean() - pred_mean[70:80].mean())
print(f"input drift across the day-80 boundary (re-baselined KS min p): {ks_at_80:.3f}")
assert ks_at_80 > 0.01, "inputs did NOT change at day 80 - the monitor is rightly silent"
print(f"prediction-mean jump at day 40: {pred_jump_40:.3f}, at day 80: {pred_jump_80:.3f}")
assert pred_jump_40 > 0.1, "covariate shift showed up in prediction distributions"
assert pred_jump_80 < 0.03, \
    "pure concept drift: inputs unchanged, predictions unchanged, accuracy in flames"

**Only labels can see pure concept drift.** And labels are late: fraud confirms in
weeks, churn in a month, loan default in a year. So the monitoring design is layered by
information speed: instant input/prediction monitors (leading, label-free, catch
covariate/pipeline events), **proxy labels** where possible (user complaints, manual
review verdicts, short-horizon behaviors correlated with the real label — nb 15's
natural labels), and true-label metrics computed the moment labels mature (lagging but
decisive). Then everything goes on one panel:

In [ ]:
LABEL_DELAY = 7
observed_acc = np.full(DAYS, np.nan)
for d in range(DAYS - LABEL_DELAY):
    observed_acc[d + LABEL_DELAY] = true_acc[d]     # day-d labels mature at d+7

acc_baseline = np.nanmean(observed_acc[30:40])
acc_alarms = np.where(observed_acc < acc_baseline - 0.12)[0]
first_acc_alarm = int(acc_alarms[acc_alarms >= 40].min())

fig, axes = plt.subplots(3, 1, figsize=(9, 7), sharex=True)
axes[0].semilogy(np.minimum(ks_pvals[:, 0], ks_pvals[:, 1]), color="#1565c0")
axes[0].axhline(0.005, color="#c62828", linestyle=":")
axes[0].set_ylabel("input drift\n(min KS p)")
axes[1].plot(pred_mean, color="#6a1b9a")
axes[1].set_ylabel("mean\nprediction")
axes[2].plot(true_acc, color="#bdbdbd", label="true accuracy (unknowable live)")
axes[2].plot(observed_acc, color="#2e7d32", label=f"observed (labels {LABEL_DELAY}d late)")
axes[2].set_ylabel("accuracy")
axes[2].legend(loc="lower left", fontsize=8)
for ax in axes:
    ax.axvline(40, color="#ef6c00", alpha=0.5)
    ax.axvline(80, color="#c62828", alpha=0.5)
axes[0].set_title("The monitoring panel: day 40 = input alarm, day 80 = only labels know")
axes[2].set_xlabel("day")
plt.show()

print(f"input alarm: day {first_input_alarm}; accuracy alarm: day {first_acc_alarm} "
      f"(the drift began day 80)")
assert first_acc_alarm >= 80 + LABEL_DELAY, \
    "the label delay is a hard floor on detecting concept drift"
assert first_acc_alarm <= 80 + LABEL_DELAY + 3, "...but it fires promptly once labels land"

Between day 80 and day 87 the system served garbage with green dashboards — the
irreducible cost of label delay, and the number that decides your retraining cadence and
label-pipeline investment (nb 20 picks this up directly). Complete the panel in practice
with: volume, missing-value rates and schema checks per feature (the pipeline-failure
impostor), latency (nb 18), calibration drift (nb 17), and *slices* of all of the above
(nb 17's lesson — drift often lands on one segment first).

## 4. Degenerate feedback loops — when the model drifts the world

The final twist: the model's own outputs shape its future training data. A recommender
shows what it predicts people want; people can only click what they're shown; the clicks
become training data that confirm the recommendation. **Exposure becomes destiny** — the
system's early guesses calcify into "truth". Simulate the spiral:

In [ ]:
def gini(x: np.ndarray) -> float:
    """Inequality of exposure: 0 = everyone equal, ->1 = one item takes all."""
    x = np.sort(x.astype(float))
    n = len(x)
    return float((2 * np.arange(1, n + 1) - n - 1) @ x / (n * x.sum() + 1e-12))

def run_platform(epsilon: float, rounds: int = 60, n_items: int = 60,
                 slots: int = 8, users_per_round: int = 200, seed: int = 7):
    """Each round: recommend `slots` items, observe clicks (true quality decides),
    retrain the CTR estimate on OUR OWN logged exposures. epsilon = exploration."""
    r = np.random.default_rng(seed)
    quality = r.beta(2, 5, n_items)                   # the truth nobody sees
    clicks = np.zeros(n_items)
    shows = np.zeros(n_items)
    for _ in range(rounds):
        # A (realistically) pessimistic prior: unshown items score low. Note
        # what this does: an item never shown can never raise its estimate.
        ctr_est = (clicks + 0.2) / (shows + 2)
        ranked = np.argsort(-ctr_est)
        for _ in range(users_per_round):
            if r.random() < epsilon:                  # exploration: random slate
                slate = r.choice(n_items, slots, replace=False)
            else:                                     # exploitation: top-predicted
                slate = ranked[:slots]
            shown = slate[r.integers(0, slots)]       # user examines one slot
            shows[shown] += 1
            clicks[shown] += r.random() < quality[shown]
    return quality, shows, clicks

quality, shows_exploit, _ = run_platform(epsilon=0.0)
_, shows_explore, _ = run_platform(epsilon=0.15)

corr_exploit = np.corrcoef(quality, shows_exploit)[0, 1]
corr_explore = np.corrcoef(quality, shows_explore)[0, 1]
top_exploit = int(np.argmax(shows_exploit))
print(f"exposure Gini - pure exploitation: {gini(shows_exploit):.2f}, "
      f"with 15% exploration: {gini(shows_explore):.2f}")
print(f"corr(quality, exposure) - exploitation: {corr_exploit:.2f}, "
      f"exploration: {corr_explore:.2f}")
print(f"most-shown item under exploitation: quality rank "
      f"{int((quality > quality[top_exploit]).sum()) + 1} of {len(quality)}")

assert gini(shows_exploit) > gini(shows_explore) + 0.1, \
    "the closed loop concentrates exposure far beyond what quality justifies"
assert corr_explore > corr_exploit + 0.3, \
    "exploration keeps the training data connected to reality; exploitation measured itself"
assert int((quality > quality[top_exploit]).sum()) + 1 > 10, \
    "the item the spiral crowned is thoroughly mediocre - early luck, calcified"

The exploitation-only platform is not measuring the world; it is measuring **its own
earlier opinions**. Ranks freeze on early luck; items never shown can never prove
themselves (the *selective labels* problem — a loan model that denies an applicant never
learns whether they'd have repaid). The cures all inject independence between the model
and its training data: **exploration** (the ε above; bandits do it optimally — nb 20),
**position/propensity corrections** (log *why* something was shown and reweight — nb 15
exercise 3), and holdout slices served randomly as a permanent truth serum. If your
model influences its own future inputs — recommendations, pricing, moderation, credit —
this loop is not an edge case; it *is* the system.

## 5. Exercises

1. **Triage.** Classify and give the first check for: (a) accuracy dips every December;
   (b) a new mobile app version launches and input distributions jump; (c) the
   fraud-rate metric doubles in an hour while inputs look stable; (d) `avg_basket_size`
   suddenly becomes 0 for 30% of traffic.
2. **Label delay design.** Chargebacks confirm fraud 30–60 days late. Design the layered
   monitoring for a fraud model: name two label-free monitors, one proxy label with its
   bias, and how you'd know the proxy itself drifted.
3. **PSI by hand.** Reference deciles each hold 10% of traffic; today's traffic puts 25%
   in the top reference decile and 5% in the bottom, others unchanged. Compute the PSI
   contribution of those two bins and give the verdict. Why is PSI *on the model's
   prediction scores* the single best default monitor?
4. **Alert arithmetic.** 200 features × daily KS tests at p<0.05: how many false alarms
   per day at perfect stability? Name three remedies and the failure mode of each.
5. **Selective labels.** The credit model only observes repayment for *approved* loans.
   Explain how this biases retraining, why it's the feedback loop of §4 in disguise, and
   two mitigation designs (one statistical, one product).

### Solutions

**1.** (a) Recurring drift (seasonality) — check: does a model trained on last December
fix it? then encode season or schedule seasonal models. (b) Covariate shift with a known
cause — check slice metrics for the new version before touching the model (maybe only
the app changed a unit — pipeline bug in disguise, nb 16's skew). (c) Label shift (or an
attack): prevalence moved with P(X) stable-looking — check the *score distribution* and
recalibrate (nb 17 ex. 2); confirm upstream labels aren't broken. (d) Pipeline failure,
full stop — nb 14's audit and schema checks before any drift theory.

**2.** Label-free: PSI on prediction scores + domain classifier on features
(weekly). Proxy: manual-review analyst verdicts on flagged transactions — biased toward
*flagged* traffic (selective labels again), so also review a small random sample
(exploration!) to keep an unbiased slice. Proxy drift check: track agreement between
matured chargebacks and the analyst verdicts on the same transactions; if agreement
decays, the proxy is drifting.

**3.** Top bin: (0.25−0.10)·ln(0.25/0.10) = 0.137; bottom: (0.05−0.10)·ln(0.05/0.10) =
0.035; total ≥ 0.17 → "investigate" band, nearly "shifted". Prediction-score PSI is the
best single monitor because the score is the model's own one-dimensional summary of
*everything it cares about* — any input change that matters to the model must pass
through it (and §3 showed its blind spot: P(Y|X)-only changes).

**4.** 200 × 0.05 = **10 false alarms every day** — a pager nobody believes within a
week. Remedies: Bonferroni/FDR correction (blunts sensitivity to small real shifts);
monitor only the model's top-k important features (misses shifts in interactions —
§2's flip); aggregate into one statistic (domain-classifier AUC or score-PSI) with one
threshold (loses per-feature attribution — keep the per-feature view for *diagnosis*,
not alerting).

**5.** Approved loans are a quality-biased sample (the model *chose* them); retraining
on them teaches "everyone repays" near the approval boundary and pushes it looser or
freezes historical prejudices — the §4 loop where exposure = approval. Statistical:
propensity weighting / treat approval as the logged policy and correct (off-policy
evaluation). Product: approve a small randomized band near the threshold (exploration
budget with a real cost, priced like nb 13's economics) to keep an unbiased label
stream.

## 6. Takeaways

- Degradation taxonomy: covariate (P(X)) — visible instantly in inputs and predictions;
  label (P(Y)) — recalibrate first; concept (P(Y|X)) — **invisible without labels**;
  and most "drift" pages are pipeline failures — check plumbing first.
- Detector toolkit: KS/PSI per feature (cheap, marginal-only), domain classifier (sees
  joint shifts + tells you what moved), prediction-score PSI (best single default).
  Layer by information speed: inputs now, proxies soon, labels late.
- **Label delay is a hard floor** on seeing concept drift — budget it, shrink it (proxy
  labels), and let it set your retraining cadence.
- If the model shapes its training data, expect the popularity spiral: exposure
  calcifies into truth. Buy truth with exploration; log propensities; keep a random
  holdout slice forever.

**Further reading.** DMLS ch. 8; Rabanser et al., *Failing Loudly: An Empirical Study
of Methods for Detecting Dataset Shift* (NeurIPS 2019); Chaney et al., *How Algorithmic
Confounding in Recommendation Systems Increases Homogeneity...* (RecSys 2018).

**Next:** [20 · Continual learning & testing in production](20-continual-learning-testing.ipynb)
— the monitors fired. Now: retrain safely, and prove the new model is actually better
*in the world*, not just offline.